In [1]:
import pandas as pd
import numpy as np

# Cargar el dataset
df = pd.read_csv('cleaned_dataset (1).csv')
print("Primeras filas del dataset:")
display(df.head())

Primeras filas del dataset:


,Pregnancies,Glucose,Blood Pressure,Skin Thickness,Insulin,BMI,Diabetes Pedigree Function,Age,Outcome
0,0,129,110,46,130,67.1,0.319,26,1
1,0,180,78,63,14,59.4,2.420,25,1
2,3,123,100,35,240,57.3,0.880,22,0
3,1,88,30,42,99,55.0,0.496,26,1
4,0,162,76,56,100,53.2,0.759,25,1


In [3]:
subset = df.sample(40, random_state=42)
train_subset = subset.iloc[:20]
test_subset = subset.iloc[20:]

In [4]:
def euclidean_distance(p1, p2):
    # Fórmula: sqrt(sum((p1_i - p2_i)^2))
    return np.sqrt(np.sum((np.array(p1) - np.array(p2))**2))

# Ejemplo proporcionado
v1 = [1, 106, 70, 28, 135, 34.2, 0.142, 22]
v2 = [2, 102, 86, 36, 120, 45.5, 0.127, 23]
print(f"Distancia calculada: {euclidean_distance(v1, v2):.4f}")

Distancia calculada: 26.2810


In [5]:
from collections import Counter

def knn_predict(test_point, train_data, k=3):
    distances = []
    for i, row in train_data.iterrows():
        # Calculamos distancia ignorando la columna 'Outcome'
        dist = euclidean_distance(test_point[:-1], row[:-1])
        distances.append((dist, row['Outcome']))

    distances.sort(key=lambda x: x[0]) # Ordenar por distancia
    neighbors = [d[1] for d in distances[:k]]
    return Counter(neighbors).most_common(1)[0][0] # Voto mayoritario

# Tabla comparativa
results = []
for _, row in test_subset.iterrows():
    pred = knn_predict(row, train_subset)
    results.append({'Real': row['Outcome'], 'Predicho': pred})
print(pd.DataFrame(results).head(10))

   Real  Predicho
0   1.0       0.0
1   0.0       0.0
2   1.0       1.0
3   1.0       1.0
4   1.0       1.0
5   1.0       1.0
6   0.0       0.0
7   1.0       0.0
8   0.0       0.0
9   0.0       0.0


In [6]:
from sklearn.model_selection import train_test_split

X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=3)

# 1. Crudo
knn.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn.predict(X_test))

# 2. Min-Max
scaler_mm = MinMaxScaler()
X_tr_mm, X_te_mm = scaler_mm.fit_transform(X_train), scaler_mm.transform(X_test)
knn.fit(X_tr_mm, y_train)
acc_mm = accuracy_score(y_test, knn.predict(X_te_mm))

# 3. Z-score
scaler_z = StandardScaler()
X_tr_z, X_te_z = scaler_z.fit_transform(X_train), scaler_z.transform(X_test)
knn.fit(X_tr_z, y_train)
acc_z = accuracy_score(y_test, knn.predict(X_te_z))

In [8]:
resultados_finales = pd.DataFrame({
    'Método': ['Sin escalar', 'Min-Max', 'Z-score'],
    'Accuracy': [acc_raw, acc_mm, acc_z]
})
display(resultados_finales)

,Método,Accuracy
0,Sin escalar,0.810127
1,Min-Max,0.734177
2,Z-score,0.746835
